# Weekly project

Today you are going to implement the last parts of the algorithm you started on monday. For reference you can see it below.

![title](algorithm_3.png)

It is a good idea to follow and track the steps in the algorithm in the below implementation. Only take one step at a time.

Once you have the algorithm up and running you can try with a larger dataset to see if your algorithm is able to maintain good accurracy over a longer distance. The larger dataset can be found here:
[Left images](https://dtudk-my.sharepoint.com/:u:/g/personal/evanb_dtu_dk/EQu8kmGBDDROtGJ7IkZB2tQBJrxmgY9t8LVM_JuEi83TYw)
[Right images](https://dtudk-my.sharepoint.com/:u:/g/personal/evanb_dtu_dk/EcKI_zrXTvpMulizidCZm4oBLJcQ_LTV9Zs6oQFF74JTRQ)

In [1]:
import numpy as np
import cv2 as cv2
import glob

def getK():
    return np.array([[7.188560e+02, 0.000000e+00, 6.071928e+02],
                     [0, 7.188560e+02, 1.852157e+02],
                     [0, 0, 1]])

def getTruePose():
    file = '00.txt'
    return np.genfromtxt(file, delimiter=' ', dtype=None)

<details>
  <summary>tips:</summary>

- [Feature Matching](https://docs.opencv.org/4.x/dc/dc3/tutorial_py_matcher.html)
- To get the keypoint of a certain match do: ```kp1[match.queryIdx].pt``` and ```kp2[match.trainIdx].pt```

In [2]:
def extract_keypoints_sift(img1, img2, K, baseline):
    """
    Use SIFT to detect keypoints and compute descriptors in both images,
    and find matches between them using knnMatch with k=2
    """
    # SIFT detector + descriptor
    sift = cv2.SIFT_create()

    # Detect & compute
    kp1, des1 = sift.detectAndCompute(img1, None)
    kp2, des2 = sift.detectAndCompute(img2, None)

    # BF matcher with L2 norm for SIFT
    bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
    matches = bf.knnMatch(des1, des2, k=2)

    # ----- Lowe's ratio test -----
    match_points1 = []
    match_points2 = []
    ratio_thresh = 0.75

    for m, n in matches:
        if m.distance < ratio_thresh * n.distance:
            pt1 = kp1[m.queryIdx].pt  # (x, y) in img1
            pt2 = kp2[m.trainIdx].pt  # (x, y) in img2
            match_points1.append(pt1)
            match_points2.append(pt2)

    # Transform best keypoints to numpy arrays
    p1 = np.array(match_points1, dtype=np.float32)  # left image
    p2 = np.array(match_points2, dtype=np.float32)  # right image

    ##### ############# ##########
    ##### Do Triangulation #######
    ##### ########################
    # projection matrix for Left and Right Image
    M_left = K.dot(np.hstack((np.eye(3), np.zeros((3, 1)))))
    M_rght = K.dot(np.hstack((np.eye(3), np.array([[-baseline, 0, 0]]).T)))

    p1_flip = np.vstack((p1.T, np.ones((1, p1.shape[0]))))
    p2_flip = np.vstack((p2.T, np.ones((1, p2.shape[0]))))

    P = cv2.triangulatePoints(M_left, M_rght, p1_flip[:2], p2_flip[:2])

    # Normalize homogeneous coordinates
    P = P / P[3]
    land_points = P[:3]           # (3, N)

    return land_points.T, p1      # (N,3), (N,2)


In [3]:
def featureTracking(prev_img, next_img, prev_points, world_points):
    """
    Use OpenCV to find prev_points from prev_img in next_img.
    Remove points that could not be tracked (status == 0)
    """
    params = dict(winSize=(21, 21),
                  maxLevel=3,
                  criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT,
                           30, 0.01))

    # Ensure correct shape & dtype
    prev_points = prev_points.reshape(-1, 1, 2).astype(np.float32)

    next_points, status, _ = cv2.calcOpticalFlowPyrLK(
        prev_img, next_img, prev_points, None, **params
    )

    status = status.reshape(-1).astype(bool)

    # Keep only successfully tracked points
    world_points = world_points[status]
    prev_points = prev_points[status].reshape(-1, 2)
    next_points = next_points[status].reshape(-1, 2)

    return world_points, prev_points, next_points


In [4]:
def playImageSequence(l_img_paths, r_img_paths, K):
    baseline = 0.54

    # First stereo pair
    left_img  = cv2.imread(l_img_paths[0], 0)
    right_img = cv2.imread(r_img_paths[0], 0)

    ##### ################################# #######
    ##### Get 3D points Using Triangulation #######
    ##### #########################################
    """
    Step 1.2 & 1.3:
    - Extract & match stereo features
    - Triangulate them to get initial 3D landmarks
    """
    landmark_3D, reference_2D = extract_keypoints_sift(left_img, right_img, K, baseline)

    # Reference image (at k-1)
    reference_img = left_img

    # Ground truth for plotting
    truePose = getTruePose()
    traj = np.zeros((600, 600, 3), dtype=np.uint8)
    maxError = 0

    for i in range(1, 101):
        print('image: ', i)

        # Current stereo pair
        curImage   = cv2.imread(l_img_paths[i], 0)
        curImage_R = cv2.imread(r_img_paths[i], 0)

        ##### ############################################################# #######
        ##### Calculate 2D and 3D feature correspondences in t=k-1 and t=k #######
        ##### #####################################################################
        """
        Step 2.2:
        Track previous 2D features from reference_img into curImage.
        This gives us 3D–2D correspondences: (landmark_3D_tr ↔ tracked_2Dpoints)
        """
        landmark_3D_tr, reference_2D_tr, tracked_2Dpoints = featureTracking(
            reference_img, curImage, reference_2D, landmark_3D
        )

        # If too few points survived tracking, skip this frame
        if len(landmark_3D_tr) < 6:
            print("  Not enough tracked points ({}), skipping frame.".format(len(landmark_3D_tr)))
            reference_img = curImage
            continue

        ##### ####################################### #######
        ##### Calculate relative pose using PNPRansac #######
        ##### ###############################################
        """
        Step 2.3:
        Estimate camera pose using PnP + RANSAC with 3D–2D matches.
        """
        obj_pts = landmark_3D_tr.astype(np.float32)                       # (N,3)
        img_pts = tracked_2Dpoints.reshape(-1, 1, 2).astype(np.float32)   # (N,1,2)

        ok, rvec, tvec, inliers = cv2.solvePnPRansac(
            objectPoints=obj_pts,
            imagePoints=img_pts,
            cameraMatrix=K,
            distCoeffs=None,
            iterationsCount=100,
            reprojectionError=8.0,
            flags=cv2.SOLVEPNP_ITERATIVE
        )

        if not ok:
            print("  PnP failed on frame {}, skipping.".format(i))
            reference_img = curImage
            continue

        ##### ####################################################### #######
        ##### Get Pose and Tranformation Matrix in world coordionates #######
        ##### ###############################################################
        rot, _ = cv2.Rodrigues(rvec)
        # Camera pose in world coordinates:
        tvec = -rot.T.dot(tvec)          # XYZ of camera wrt world
        inv_transform = np.hstack((rot.T, tvec))  # [R_wc | t_wc]

        ##### ################################# #######
        ##### Get 3D points Using Triangulation #######
        ##### #########################################
        """
        Step 2.4:
        Re-triangulate NEW stereo features at time k (curImage, curImage_R),
        then lift them from camera frame to world frame.
        """
        landmark_3D_new, reference_2D_new = extract_keypoints_sift(
            curImage, curImage_R, K, baseline
        )

        # Project the new 3D points from camera to world coordinates
        landmark_3D_h = np.vstack((landmark_3D_new.T, np.ones((1, landmark_3D_new.shape[0]))))
        landmark_3D_world = inv_transform.dot(landmark_3D_h).T   # (N,3)

        # Update reference sets for next iteration
        reference_2D = reference_2D_new.astype('float32')
        landmark_3D  = landmark_3D_world
        reference_img = curImage

        ##### ################################## #######
        ##### START OF Print and visualize stuff #######
        ##### ##########################################
        draw_x, draw_y = int(tvec[0]) + 300, 600 - (int(tvec[2]) + 100)
        true_x, true_y = int(truePose[i][3]) + 300, 600 - (int(truePose[i][11]) + 100)

        curError = np.sqrt(
            (tvec[0] - truePose[i][3]) ** 2 +
            (tvec[1] - truePose[i][7]) ** 2 +
            (tvec[2] - truePose[i][11]) ** 2
        )
        maxError = max(maxError, curError)

        print(tvec[0], tvec[1], tvec[2], rvec[0], rvec[1], rvec[2])
        print([truePose[i][3], truePose[i][7], truePose[i][11]])

        text = "Coordinates: x ={0:02f}m y = {1:02f}m z = {2:02f}m".format(
            float(tvec[0]), float(tvec[1]), float(tvec[2])
        )

        cv2.circle(traj, (draw_x, draw_y), 1, (0, 0, 255), 2)
        cv2.circle(traj, (true_x, true_y), 1, (255, 0, 0), 2)
        cv2.rectangle(traj, (10, 30), (550, 50), (0, 0, 0), cv2.FILLED)
        cv2.putText(traj, text, (10, 50), cv2.FONT_HERSHEY_PLAIN, 1,
                    (255, 255, 255), 1, 8)

        h1, w1 = traj.shape[:2]
        h2, w2 = curImage.shape[:2]
        vis = np.zeros((max(h1, h2), w1 + w2, 3), np.uint8)
        vis[:h1, :w1, :3] = traj
        vis[:h2, w1:w1 + w2, :3] = np.dstack((np.dstack((curImage, curImage)), curImage))

        cv2.imshow("Trajectory", vis)
        k = cv2.waitKey(1) & 0xFF
        if k == 27:
            break

    cv2.waitKey(0)
    cv2.destroyAllWindows()
    print('Maximum Error: ', maxError)
    ##### ################################ #######
    ##### END OF Print and visualize stuff #######
    ##### ########################################


In [5]:
# Load image paths
left_img_paths = sorted(glob.glob('left/*.png'))
right_img_paths = sorted(glob.glob('right/*.png'))

K = getK()

playImageSequence(left_img_paths, right_img_paths, K)

image:  1


C:\Users\UPASANA\AppData\Local\Temp\ipykernel_57956\1146414841.py:108: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  draw_x, draw_y = int(tvec[0]) + 300, 600 - (int(tvec[2]) + 100)
C:\Users\UPASANA\AppData\Local\Temp\ipykernel_57956\1146414841.py:122: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  float(tvec[0]), float(tvec[1]), float(tvec[2])


[-0.00227638] [-0.00703883] [0.67547857] [-0.0021743] [0.0033434] [-0.00261038]
[np.float64(-0.04690294), np.float64(-0.02839928), np.float64(0.8586941)]
image:  2
[-0.01421142] [-0.01295007] [1.37229914] [-0.00388048] [0.00737935] [-0.00082553]
[np.float64(-0.09374345), np.float64(-0.05676064), np.float64(1.716275)]
image:  3
[-0.03120016] [-0.02015722] [2.08366938] [-0.00540022] [0.01121703] [-0.00063776]
[np.float64(-0.1406429), np.float64(-0.08515762), np.float64(2.574964)]
image:  4
[-0.04991075] [-0.03387803] [2.8142021] [-0.00587348] [0.01610271] [0.00107941]
[np.float64(-0.1874858), np.float64(-0.1135202), np.float64(3.432648)]
image:  5
[-0.06623029] [-0.04580844] [3.5570171] [-0.00606627] [0.02065917] [6.16482809e-05]
[np.float64(-0.2343818), np.float64(-0.141915), np.float64(4.291335)]
image:  6
[-0.09174804] [-0.05829521] [4.31238531] [-0.007368] [0.02521116] [0.00215092]
[np.float64(-0.2812195), np.float64(-0.1702743), np.float64(5.148987)]
image:  7
[-0.12515105] [-0.0704

# Challenge 
The current implementation only uses features computed at the current timestep. However, as we process more images we potentially have a lot of features from previous timesteps that are still valid. The challenge is to expand the `extract_keypoints_surf(..., refPoints)` function by giving it old reference points. You should then combine your freshly computed features with the old features and remove all duplicates. This requires you to keep track of old features and 3D points.

Hint 1: The following function `removeDuplicate` can be used for removing duplicates.

Hint 2: you are not interested in points that are behind you, so remember to remove points that are negative in the direction you move.

In [8]:
def removeDuplicate(queryPoints, refPoints, radius=5):
    """
    Remove query points that lie within 'radius' pixels of any existing reference points.
    Returns a boolean mask of which queryPoints are kept.
    """
    # Boolean mask of points to keep
    keep_mask = np.ones(len(queryPoints), dtype=bool)

    for i, q in enumerate(queryPoints):

        # Bounding box for duplicate detection
        x_low, x_high = q[0] - radius, q[0] + radius
        y_low, y_high = q[1] - radius, q[1] + radius

        # First filter by X distance
        mask_x = (refPoints[:, 0] >= x_low) & (refPoints[:, 0] <= x_high)
        ref_in_x = refPoints[mask_x]

        if len(ref_in_x) == 0:
            continue

        # Then filter by Y distance
        mask_y = (ref_in_x[:, 1] >= y_low) & (ref_in_x[:, 1] <= y_high)
        ref_in_xy = ref_in_x[mask_y]

        # If both conditions satisfied → duplicate
        if len(ref_in_xy) > 0:
            keep_mask[i] = False

    return keep_mask
